# RetailMart Lakehouse Project

## Bronze Layer – Payments Data Ingestion

### Objective
This notebook ingests the raw Payments dataset into the Bronze Layer of the RetailMart Lakehouse.
The Bronze layer preserves the source data in its original form while adding ingestion metadata for auditability and traceability.

### Source
raw_payments_dataset.csv

### Target
retailmart.bronze.payments

### Author
Kashish Soni

### Layer
Bronze

In [0]:
%run ../00_Setup/01_Config

In [0]:
%run ../Utils/Common_Utils

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)

In [0]:
PIPELINE_NAME = "Bronze_Payments_Load"
# Declared in 01_Config
SOURCE_FILE = RAW_PAYMENTS
TARGET_TABLE = BRONZE_PAYMENTS
# generate_run_id - declared in common_util functions
RUN_ID = generate_run_id()
START_TIME = start_pipeline()

print(f"Pipeline : {PIPELINE_NAME}")
print(f"Run ID   : {RUN_ID}")

Pipeline Started : 2026-07-15 06:01:36.400502
Pipeline : Bronze_Payments_Load
Run ID   : 3532582a-9b9d-4aa6-9775-be38ea50d344


In [0]:
display(
    spark.read
         .option("header", True)
         .csv(SOURCE_FILE)
         .limit(10)
)

order_id,payment_sequential,payment_type,payment_installments,payment_value
ORD_0000001,1,voucher,12,2775.42
ORD_0000001,2,credit_card,1,32.43
ORD_0000002,1,voucher,1,805.73
ORD_0000003,1,credit_card,6,508.88
ORD_0000004,1,boleto,2,1589.64
ORD_0000004,2,boleto,3,35.07
ORD_0000005,1,credit_card,3,1592.91
ORD_0000006,1,credit_card,1,495.97
ORD_0000007,1,credit_card,2,1738.92
ORD_0000008,1,voucher,1,1107.69


In [0]:
spark.read.option("header", True).csv(SOURCE_FILE).printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



In [0]:
# Explicit Schema 

payments_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("payment_sequential", IntegerType(), True),
    StructField("payment_type", StringType(), True),
    StructField("payment_installments", IntegerType(), True),
    StructField("payment_value", DoubleType(), True)
])

In [0]:
payments_df = (
    spark.read
         .schema(payments_schema)
         .option("header", True)
         .csv(SOURCE_FILE)
)

In [0]:
# Print dataset profile - declared in common_util functions
dataset_profile(
    payments_df,
    "PAYMENTS DATASET PROFILE"
)

PAYMENTS DATASET PROFILE
Rows    : 57388
Columns : 5
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



order_id,payment_sequential,payment_type,payment_installments,payment_value
ORD_0000001,1,voucher,12,2775.42
ORD_0000001,2,credit_card,1,32.43
ORD_0000002,1,voucher,1,805.73
ORD_0000003,1,credit_card,6,508.88
ORD_0000004,1,boleto,2,1589.64
ORD_0000004,2,boleto,3,35.07
ORD_0000005,1,credit_card,3,1592.91
ORD_0000006,1,credit_card,1,495.97
ORD_0000007,1,credit_card,2,1738.92
ORD_0000008,1,voucher,1,1107.69


In [0]:
expected_schema = {
    "order_id": "string",
    "payment_sequential": "int",
    "payment_type": "string",
    "payment_installments": "int",
    "payment_value": "double"
}

In [0]:
schema_status = validate_schema(
    payments_df,
    payments_schema
)

Schema Validation Passed


In [0]:
pk_status = validate_primary_key(
    payments_df,
    ["order_id", "payment_sequential"]
)

Total Rows : 57388
Distinct Count : 57388
Primary Key Validation Passed — (order_id, payment_sequential)


In [0]:
total_rows = payments_df.count()
print(f"Rows    : {total_rows}")
print(f"Columns : {len(payments_df.columns)}")
payments_df.printSchema()
display(payments_df.limit(10))

Rows    : 57388
Columns : 5
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)



order_id,payment_sequential,payment_type,payment_installments,payment_value
ORD_0000001,1,voucher,12,2775.42
ORD_0000001,2,credit_card,1,32.43
ORD_0000002,1,voucher,1,805.73
ORD_0000003,1,credit_card,6,508.88
ORD_0000004,1,boleto,2,1589.64
ORD_0000004,2,boleto,3,35.07
ORD_0000005,1,credit_card,3,1592.91
ORD_0000006,1,credit_card,1,495.97
ORD_0000007,1,credit_card,2,1738.92
ORD_0000008,1,voucher,1,1107.69


In [0]:
duplicate_rows = duplicate_summary(
    payments_df,
    total_rows
)

Duplicate Rows : 0


In [0]:
null_summary(payments_df)

order_id,payment_sequential,payment_type,payment_installments,payment_value
0,0,0,0,0


In [0]:
# Invalid payment record
invalid_payment = payments_df.filter(
    F.col("payment_value") <= 0
).count()

print(f"Invalid Payment Amount : {invalid_payment}")

Invalid Payment Amount : 0


In [0]:
# Invalid instalments
invalid_installments = payments_df.filter(
    F.col("payment_installments") <= 0
).count()

print(f"Invalid Installments : {invalid_installments}")

Invalid Installments : 0


In [0]:
# Payment method distributions
display(
    payments_df
    .groupBy("payment_type")
    .count()
    .orderBy(F.desc("count"))

)

payment_type,count
credit_card,28563
voucher,9663
debit_card,9631
boleto,9531


In [0]:
# Orders with multiple payments
display(
    payments_df
    .groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(20)

)

order_id,count
ORD_0000146,2
ORD_0000175,2
ORD_0000390,2
ORD_0000472,2
ORD_0000506,2
ORD_0000534,2
ORD_0000678,2
ORD_0000843,2
ORD_0000904,2
ORD_0000910,2


# Audit Columns

payments_df = (
    payments_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("ingestion_date", F.current_date())
    .withColumn("pipeline_name", F.lit(PIPELINE_NAME))
    .withColumn("run_id", F.lit(RUN_ID))
)

In [0]:
payments_df = add_audit_columns(
    payments_df,
    PIPELINE_NAME,
    RUN_ID
)

In [0]:
status, error = write_bronze_table(
    payments_df,
    TARGET_TABLE
)

Bronze table written: retailmart.bronze.payments


In [0]:
bronze_df = spark.table(TARGET_TABLE)

rows_written = bronze_df.count()

print(f"Rows Written : {rows_written}")

Rows Written : 57388


In [0]:
bronze_load_report(
    pipeline_name=PIPELINE_NAME,
    run_id=RUN_ID,
    source=SOURCE_FILE,
    target=TARGET_TABLE,
    rows_read=total_rows,
    rows_written=rows_written,
    duplicate_count=duplicate_rows,
    start_time=START_TIME,
    status=status,
)

BRONZE LOAD REPORT
Pipeline        : Bronze_Payments_Load
Run ID          : 3532582a-9b9d-4aa6-9775-be38ea50d344
Source          : /Volumes/dbacademy/default/raw/raw_payments_dataset.csv
Target          : retailmart.bronze.payments
Rows Read       : 57388
Rows Written    : 57388
Duplicate Rows  : 0
Start Time      : 2026-07-15 06:01:36.400502
End Time        : 2026-07-15 06:01:57.998147
Duration (sec)  : 21.6
Status          : SUCCESS
Error:          : None


In [0]:
validation_summary = spark.createDataFrame(
[
    ("Schema Validation", "PASS"),
    ("Composite PK Validation",
     "PASS" if pk_status else "FAIL"),
    ("Duplicate Validation",
     "PASS" if duplicate_rows else "FAIL"),
    ("Null Analysis", "PASS"),
    ("Payment Value Validation",
     "PASS" if invalid_payment == 0 else "FAIL"),
    ("Installment Validation",
     "PASS" if invalid_installments == 0 else "FAIL")
],
["Validation", "Status"]
)

display(validation_summary)

Validation,Status
Schema Validation,PASS
Composite PK Validation,PASS
Duplicate Validation,FAIL
Null Analysis,PASS
Payment Value Validation,PASS
Installment Validation,PASS
